# Кибериммунная автономность$\\$Создание конструктивно защищённого автономного наземного транспортного средства$\\$Модуль 4

## О документе

Версия 1.03

Модуль 4 для регионального этапа соревнований по кибериммунной автономности

### Модуль 4. Следование по трассе с киберпрепятствиями

Для успешного выполнения этого задания необходимое активировать специальный режим работы виртуальной машинки - в рамках этого задания можно менять только блоки, отвечающие за безопасность (ограничитель, монитор безопасности), **другие блоки менять запрещено**.

При прохождении маршрута будут имитироваться атаки со стороны злоумышленников, которые будут пытаться нарушить цели безопасности. Важно, чтобы им это не удалось.

1. Запустите свою машинку и убедитесь, что она проходит всю трассу без нарушений ограничений скорости. При необходимости измените логику работы блока безопасности
2. Добавьте контроль доставки груза в модуле SafetyBlock - убедитесь, что груз доставляется до конечной точки маршрута.
3. Если в модуле 3 вы реализовали монитор безопасности - не забудьте его перенести в этот модуль, это принесёт дополнительные баллы!

Активация киберпрепятствий в системе управления:

после инициализации системы управления добавьте следующую строку
```python
control_system.enable_surprises()
```

В этом блоке добавьте все ваши реализации изменённых бортовых систем

In [3]:
import json
from pathlib import Path
from src.blackbox import BaseBlackBox
from src.event_types import Event
from src.crypto_module import verify_signature, InvalidSignature
class BlackBox(BaseBlackBox):
    """Реализация черного ящика для безопасного хранения событий"""
    
    def __init__(self, storage_path: str = "blackbox.log", public_key_pem: str = None):
        super().__init__()
        self.storage_path = Path(storage_path)
        self.public_key_pem = public_key_pem
        
        # Очищаем файл при инициализации
        with open(self.storage_path, 'w') as f:
            f.write("")
    
    def _log_event(self, event: Event, signature: str) -> bool:
        """Логирует событие после проверки подписи
        
        Args:
            event: событие для логирования
            signature: цифровая подпись события
            
        Returns:
            bool: True если подпись верна и событие записано, иначе False
        """
        if not self.public_key_pem:
            raise ValueError("Public key is not set")
            
        try:
            # Проверяем подпись
            is_valid = verify_signature(event, signature, self.public_key_pem)
            
            if is_valid:
                # Логируем событие
                with open(self.storage_path, 'a') as f:
                    log_entry = {
                        'event': event.__dict__,
                        'signature': signature,
                        'valid': True
                    }
                    f.write(json.dumps(log_entry) + "\n")
                return True
            else:
                # Логируем невалидное событие
                with open(self.storage_path, 'a') as f:
                    log_entry = {
                        'event': event.__dict__,
                        'signature': signature,
                        'valid': False,
                        'error': 'Invalid signature'
                    }
                    f.write(json.dumps(log_entry) + "\n")
                return False
                
        except InvalidSignature:
            # Логируем ошибку проверки подписи
            with open(self.storage_path, 'a') as f:
                log_entry = {
                    'event': event.__dict__,
                    'signature': signature,
                    'valid': False,
                    'error': 'Signature verification failed'
                }
                f.write(json.dumps(log_entry) + "\n")
            return False
        except Exception as e:
            # Логируем другие ошибки
            with open(self.storage_path, 'a') as f:
                log_entry = {
                    'event': event.__dict__,
                    'signature': signature,
                    'valid': False,
                    'error': str(e)
                }
                f.write(json.dumps(log_entry) + "\n")
            return False

In [ ]:
# ваш код

# если вы собираетесь выполнить это задание, напишите в этой ячейке код выполнения изменённых блоков. 
# cам код допускается редактировать в папке src
# в случае успешной реализации машинка должна успешно пройти по маршруту, а в случае искажения маршрутного задания в блоке коммуникационного шлюза должна остаться на месте, 
# при этом именно блок безопасности должен заблокировать движение
from src.helpers import deserialize_position, serialize_position
from src.safety_block import BaseSafetyBlock
from src.config import *
import math
from time import sleep
from geopy import Point as GeoPoint
from src.config import LOG_ERROR, LOG_INFO, LOG_DEBUG
from multiprocessing import Queue
from src.config import SERVOS_QUEUE_NAME
from src.event_types import Event
from src.control_system import BaseControlSystem
from src.navigation_system import BaseNavigationSystem
from src.communication_gateway import BaseCommunicationGateway
from multiprocessing import Queue
from src.config import SAFETY_BLOCK_QUEUE_NAME
from src.config import SERVOS_QUEUE_NAME
from src.config import CONTROL_SYSTEM_QUEUE_NAME
from src.event_types import Event
from src.mission_type import *
from src.route import *
from time import sleep
from geopy import Point as GeoPoint


from src.queues_dir import QueuesDirectory
from src.servos import Servos
from src.sitl import SITL
from src.cargo_bay import CargoBay
from src.mission_planner import MissionPlanner
from src.config import LOG_ERROR, LOG_INFO
from src.mission_planner_mqtt import MissionSender
from src.mission_planner import Mission
from src.sitl_mqtt import TelemetrySender
from src.system_wrapper import SystemComponentsContainer
from src.wpl_parser import WPLParser
from src.blackbox import BaseBlackBox
from src.crypto_module import * 

# добавьте изменения сюда

from src.security_monitory import BaseSecurityMonitor
from src.security_policy_type import SecurityPolicy

from src.config import *
    
# Generate RSA keys for signing events
private_key, public_key = generate_rsa_keys()

# Create BlackBox instance
blackbox = BlackBox(storage_path="blackbox.log", public_key_pem=public_key)

class SecurityMonitor(BaseSecurityMonitor):
    """ класс монитора безопасности """

    def __init__(self, queues_dir, public_key=None, blackbox = None):
        super().__init__(queues_dir=queues_dir, public_key=public_key, blackbox=blackbox)
        self._init_set_security_policies()


    def _init_set_security_policies(self):
        """ инициализация политик безопасности """
        default_policies = [
            SecurityPolicy(
                source=COMMUNICATION_GATEWAY_QUEUE_NAME,
                destination=CONTROL_SYSTEM_QUEUE_NAME,
                operation='set_mission'),
            SecurityPolicy(
                source=COMMUNICATION_GATEWAY_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation="set_mission"),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=NAVIGATION_QUEUE_NAME,
                operation="request_position"),
            SecurityPolicy(
                source=NAVIGATION_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation="position_update"),
            SecurityPolicy(
                source=NAVIGATION_QUEUE_NAME,
                destination=CONTROL_SYSTEM_QUEUE_NAME,
                operation="position_update"),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation="set_speed"),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation="set_direction"),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation="lock_cargo"),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation="release_cargo"),            
            SecurityPolicy(
                source=SAFETY_BLOCK_QUEUE_NAME,
                destination=SERVOS_QUEUE_NAME,
                operation="set_speed"),
            SecurityPolicy(
                source=SAFETY_BLOCK_QUEUE_NAME,
                destination=SERVOS_QUEUE_NAME,
                operation="set_direction"),
            SecurityPolicy(
                source=SAFETY_BLOCK_QUEUE_NAME,
                destination=CARGO_BAY_QUEUE_NAME,
                operation="release_cargo"),
            SecurityPolicy(
                source=SAFETY_BLOCK_QUEUE_NAME,
                destination=CARGO_BAY_QUEUE_NAME,
                operation="lock_cargo"),
            SecurityPolicy(
                source=SAFETY_BLOCK_QUEUE_NAME,
                destination=CARGO_BAY_QUEUE_NAME,
                operation="lock_cargo"),
            SecurityPolicy(
                source=COMMUNICATION_GATEWAY_QUEUE_NAME,
                destination=CONTROL_SYSTEM_QUEUE_NAME,
                operation='position_update'),
            SecurityPolicy(
                source=CONTROL_SYSTEM_QUEUE_NAME,
                destination=SAFETY_BLOCK_QUEUE_NAME,
                operation="position_update"),
        ]
        self.set_security_policies(policies=default_policies)        

    def set_security_policies(self, policies):
        """ установка новых политик безопасности """
        self._security_policies = policies
        self._log_message(
            LOG_INFO, f"изменение политик безопасности: {policies}")
    
    def _proceed(self, event: Event):
        """Process verified event"""
        if self._blackbox:
            self._blackbox._log_event(event, event.signature)
            
        destination_q = self._queues_dir.get_queue(event.destination)
        if destination_q is None:
            self._log_message(LOG_ERROR, f"Recipient not found for {event}")
        else:
            destination_q.put(event)

    def _check_event(self, event: Event):
        """ проверка входящих событий """
        self._log_message(
            LOG_DEBUG, f"проверка события {event}, по умолчанию выполнение запрещено")

        authorized = False
        if not hasattr(event, 'signature'):
            self._log_message(LOG_ERROR, "Event missing signature")
            return False
            
        if not self._public_key:
            self._log_message(LOG_ERROR, "No public key configured for verification")
            return False
            
        try:
            if not verify_signature(event, event.signature, self._public_key):
                self._log_message(LOG_ERROR, "Invalid event signature")
                return False
        except Exception as e:
            self._log_message(LOG_ERROR, f"Signature verification failed: {str(e)}")
            return False
        


        request = SecurityPolicy(
            source=event.source,
            destination=event.destination,
            operation=event.operation)

        if request in self._security_policies:
            self._log_message(
                LOG_DEBUG, "событие разрешено политиками, выполняем")
            authorized = True
        
        if authorized is False:
            self._log_message(LOG_ERROR, f"событие не разрешено политиками безопасности! {event}")
        return authorized

class SafetyBlock(BaseSafetyBlock):
    """ класс ограничений безопасности """

    def __init__(self, queues_dir, private_key: str = None, log_level=DEFAULT_LOG_LEVEL):
        super().__init__(queues_dir, blackbox=None, log_level= log_level)
        self._private_key = private_key
    def _handle_position_data(self, position):
        """Universal position handler that works with any input format"""
        try:
            # Case 1: Already a GeoPoint
            if hasattr(position, 'latitude') and hasattr(position, 'longitude'):
                return position
            
            # Case 2: Serialized dictionary
            if isinstance(position, dict):
                if '__geopoint__' in position:
                    return Point(position['lat'], position['lon'], position.get('alt', 0))
                if 'latitude' in position:
                    return Point(position['latitude'], position['longitude'], position.get('altitude', 0))
            
            # Case 3: None or invalid
            self._log_message(LOG_ERROR, f"Invalid position format: {type(position)}")
            return None
            
        except Exception as e:
            self._log_message(LOG_ERROR, f"Position processing failed: {str(e)}")
            return None

    def _set_mission(self, mission: Mission):
        """ установка нового маршрутного задания """
        if not self._private_key:
            raise ValueError("Private key not set for signing")     
        self._mission = mission
        self._route = Route(points=self._mission.waypoints,
                            speed_limits=self._mission.speed_limits)
        
    def _set_new_direction(self, direction: float):
        """ установка нового направления перемещения """
        self._log_message(LOG_INFO, f"текущие координаты: {self._position}")
        self._log_message(LOG_DEBUG, f"маршрутное задание: {self._mission}")
        self._log_message(LOG_DEBUG, f"состояние маршруте: {self._route}")
        # TODO реализовать контроль безопасности изменения направления

        
        if not self._position or not self._route:
            self._log_message(LOG_ERROR, f"не получилось установить новое направление - неизвестный маршрут или позиция машинки")
            return
        
        current = self._position
        next = self._route.next_point()
        bearing = self._calculate_bearing(current, next)
        
        new_direction = max(bearing - 5, min(bearing + 5, direction))
        
        if new_direction != direction:
            self._log_message(LOG_ERROR, f"попытка изменить направление движения не в ту сторону: {direction}. Скорректировано на {new_direction}")
                
        self._direction = new_direction
        # отправка сообщения с желаемым направлением
        self._send_direction_to_consumers()

    def _set_new_speed(self, speed: float):
        """ установка новой скорости """
        if not self._route:
            self._log_message(LOG_ERROR, f"не получилось установить новую скорость - неизвестный маршрут")
            return

        new_speed = max(0, min(speed, self._route.calculate_speed()))
        if new_speed != speed:
            self._log_message(LOG_ERROR, f"попытка изменить скорость движения на запрещенную: {speed}. Скорректировано на {new_speed}")
                
        self._speed = new_speed
        self._send_speed_to_consumers()

    def _send_speed_to_consumers(self):
        self._log_message(LOG_DEBUG, "отправляем скорость получателям")
        if not self._private_key:
            raise ValueError("Private key not set for signing")
        servos_q_name = SERVOS_QUEUE_NAME
        
        security_monitor_q = self._queues_dir.get_queue(SECURITY_MONITOR_QUEUE_NAME)

        # отправка сообщения с желаемой скоростью
        event_speed = Event(source=self.event_source_name,
                            destination=servos_q_name,
                            operation="set_speed",
                            parameters=self._speed
                            )
        event_speed.signature = create_signature(event_speed, self._private_key)
        security_monitor_q.put(event_speed)

    def _send_direction_to_consumers(self):
        self._log_message(LOG_DEBUG, "отправляем направление получателям")
        if not self._private_key:
            raise ValueError("Private key not set for signing")
        servos_q_name = SERVOS_QUEUE_NAME
        security_monitor_q = self._queues_dir.get_queue(SECURITY_MONITOR_QUEUE_NAME)

        # отправка сообщения с желаемой скоростью
        event_speed = Event(source=self.event_source_name,
                            destination=servos_q_name,
                            operation="set_direction",
                            parameters=self._direction
                            )
        event_speed.signature = create_signature(event_speed, self._private_key)
        security_monitor_q.put(event_speed)

    def _lock_cargo(self, _):
        self._log_message(LOG_INFO, "Блокировка грузового отсека")
        self._send_lock_cargo_to_consumers()

    def _send_lock_cargo_to_consumers(self):
        self._log_message(LOG_DEBUG, "Отправляем команду блокировки в CargoBay")
        if not self._private_key:
            raise ValueError("Private key not set for signing")
        event = Event(
            source=self.event_source_name,
            destination=CARGO_BAY_QUEUE_NAME,
            operation="lock_cargo",
            parameters=None
        )
        event.signature = create_signature(event, self._private_key)
        security_monitor_q = self._queues_dir.get_queue(SECURITY_MONITOR_QUEUE_NAME)
        security_monitor_q.put(event)
        
    def _release_cargo(self, _):
        if not self._route or not self._route.route_finished:
            self._log_message(LOG_DEBUG, "Не разрешили открыть груз тк не на маршруте или не закончили его")
            return
        self._send_release_cargo_to_consumers()

    def _send_release_cargo_to_consumers(self):
        self._log_message(LOG_DEBUG, "Отправляем команду выгрузки в CargoBay")
        if not self._private_key:
            raise ValueError("Private key not set for signing")
        event = Event(
            source=self.event_source_name,
            destination=CARGO_BAY_QUEUE_NAME,
            operation="release_cargo",
            parameters=None
        )
        event.signature = create_signature(event, self._private_key)
        security_monitor_q = self._queues_dir.get_queue(SECURITY_MONITOR_QUEUE_NAME)
        security_monitor_q.put(event)


    def _set_new_position(self, position_data):
        """
        Установка новой позиции с обработкой всех форматов:
        - Сырой GeoPoint объект
        - Сериализованный словарь (новый и старый форматы)
        - None/некорректные данные
        """
        from geopy import Point
        
        # 1. Обработка None
        if position_data is None:
            return
            
        # 2. Если пришёл GeoPoint
        if hasattr(position_data, 'latitude'):
            self._position = position_data
            
        # 3. Если пришёл словарь
        elif isinstance(position_data, dict):
            try:
                self._position = Point(
                    position_data['latitude'],
                    position_data['longitude'],
                    position_data.get('altitude', 0)
                )
            except KeyError:
                self._log_message(LOG_ERROR, "Неверный формат позиции")
                return
        # 4. Оригинальная логика
        if not self._route:
            return
            
        distance = self._route.calculate_remaining_distance_to_next_point(self._position)
        if distance <= self._tolerance_meters:
            self._route.move_to_next_point()
            if self._route.route_finished:
                self._log_message(LOG_INFO, "Маршрут завершён")
        #I HATE CODING
        
    def _calculate_bearing(self, start, end):
        """Расчет направления с поддержкой сериализованных точек"""
        start_pt = self._handle_position_data(start)
        end_pt = self._handle_position_data(end)
        
        if not start_pt or not end_pt:
            return 0  # Default bearing
            
        # Original bearing calculation
        delta_lon = math.radians(end_pt.longitude - start_pt.longitude)
        lat1 = math.radians(start_pt.latitude)
        lat2 = math.radians(end_pt.latitude)
        
        x = math.sin(delta_lon) * math.cos(lat2)
        y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(delta_lon)
        
        bearing = math.degrees(math.atan2(x, y))
        return (bearing + 360) % 360

        
class CommunicationGateway(BaseCommunicationGateway):
    def __init__(self, queues_dir, private_key=None, log_level=DEFAULT_LOG_LEVEL):
        super().__init__(queues_dir, log_level)
        self._private_key = private_key
        
    def _send_mission_to_consumers(self):
        """ method to send mission with digital signature """
        control_q_name = CONTROL_SYSTEM_QUEUE_NAME
        safety_q_name = SAFETY_BLOCK_QUEUE_NAME
        security_monitor_q = self._queues_dir.get_queue(SECURITY_MONITOR_QUEUE_NAME)
        
        # Create and sign events
        event_control = Event(
            source=self.event_source_name,
            destination=control_q_name,
            operation="set_mission",
            parameters=self._mission
        )
        event_control.signature = create_signature(event_control, self._private_key)
        
        event_safety = Event(
            source=self.event_source_name,
            destination=safety_q_name,
            operation="set_mission",
            parameters=self._mission
        )
        event_safety.signature = create_signature(event_safety, self._private_key)
        
        security_monitor_q.put(event_control)
        security_monitor_q.put(event_safety)

class ControlSystem(BaseControlSystem):
    def __init__(self, queues_dir, private_key=None, log_level=DEFAULT_LOG_LEVEL):
        super().__init__(queues_dir, log_level)
        self._private_key = private_key
        
    def _send_speed_and_direction_to_consumers(self, speed, direction):
        safety_q_name = SAFETY_BLOCK_QUEUE_NAME
        security_monitor_q = self._queues_dir.get_queue(SECURITY_MONITOR_QUEUE_NAME)
        
        # Create and sign speed event
        event_speed = Event(
            source=self.event_source_name,
            destination=safety_q_name,
            operation="set_speed",
            parameters=speed
        )
        event_speed.signature = create_signature(event_speed, self._private_key)
        
        # Create and sign direction event
        event_direction = Event(
            source=self.event_source_name,
            destination=safety_q_name,
            operation="set_direction",
            parameters=direction
        )
        event_direction.signature = create_signature(event_direction, self._private_key)
        
        security_monitor_q.put(event_speed)
        security_monitor_q.put(event_direction)
        
    def _lock_cargo(self):
        """ заблокировать грузовой отсек """
        event = Event(source=CONTROL_SYSTEM_QUEUE_NAME,
                    destination=SAFETY_BLOCK_QUEUE_NAME,
                    operation="lock_cargo", parameters= None
                    )
        security_monitor_q = self._queues_dir.get_queue(SECURITY_MONITOR_QUEUE_NAME)
        security_monitor_q.put(event)

    def _release_cargo(self):
        """ открыть грузовой отсек """
        event = Event(source=CONTROL_SYSTEM_QUEUE_NAME, 
                    destination=SAFETY_BLOCK_QUEUE_NAME,
                    operation="release_cargo", parameters= None
                    )  
        security_monitor_q = self._queues_dir.get_queue(SECURITY_MONITOR_QUEUE_NAME)
        security_monitor_q.put(event)
        

class NavigationSystem(BaseNavigationSystem):
    def __init__(self, queues_dir, private_key=None, log_level=DEFAULT_LOG_LEVEL):
        super().__init__(queues_dir, log_level)
        self._private_key = private_key
    
    def _send_position_to_consumers(self):
        serialized_pos = serialize_geopoint(self._position)
        event = Event(
            source=self.event_source_name,
            destination=CONTROL_SYSTEM_QUEUE_NAME,
            operation="position_update",
            parameters=serialized_pos
        )
        
    def _send_position_to_consumers(self):
        control_q_name = CONTROL_SYSTEM_QUEUE_NAME
        safety_q_name = SAFETY_BLOCK_QUEUE_NAME
        security_monitor_q = self._queues_dir.get_queue(SECURITY_MONITOR_QUEUE_NAME)
        
        # Create and sign events
        event_control = Event(
            source=self.event_source_name,
            destination=control_q_name,
            operation="position_update",
            parameters=self._position
        )
        event_control.signature = create_signature(event_control, self._private_key)
        
        event_safety = Event(
            source=self.event_source_name,
            destination=safety_q_name,
            operation="position_update",
            parameters=self._position
        )
        event_safety.signature = create_signature(event_safety, self._private_key)
        
        security_monitor_q.put(event_control)
        security_monitor_q.put(event_safety)


Если у вас настроена и работает СУПА, установите в True значение переменной afcs_present

In [5]:
afcs_present = True

Поменяем идентификатор машинки для этого модуля

In [6]:
car_id = "m4"

BlackBox

В следующем блоке измените маршрут на ваш, его можно скопировать из модуля 2.
 
Для проверки работы систем безопасности будут активированы киберпрепятствия

```python
control_system.enable_critical_surprises()
```

In [7]:
# используем то же маршрутное задание, которое было в модуле 2
from time import sleep

from src.queues_dir import QueuesDirectory
from src.servos import Servos
from src.sitl import SITL
from src.cargo_bay import CargoBay
from src.mission_planner import MissionPlanner
from src.config import LOG_ERROR, LOG_INFO
from src.mission_planner_mqtt import MissionSender
from src.mission_planner import Mission
from src.sitl_mqtt import TelemetrySender
from src.system_wrapper import SystemComponentsContainer
from src.wpl_parser import WPLParser
from src.mission_type import GeoSpecificSpeedLimit


# возьмём маршрут из модуля 2
wpl_file = "module4.waypoints"

parser = WPLParser(wpl_file)    
points = parser.parse()

# обновите скоростные ограничения для вашего маршрута!
speed_limits = [
    GeoSpecificSpeedLimit(0, 20),
    GeoSpecificSpeedLimit(3, 60),
    GeoSpecificSpeedLimit(16, 110),
    GeoSpecificSpeedLimit(18, 60),
]

home = points[0]
mission = Mission(home=home, waypoints=points,speed_limits=speed_limits, armed=True)

# каталог очередей для передачи сообщений между блоками
queues_dir = QueuesDirectory() 
# создание блоков передачи данных в СУПА
if afcs_present:
    mission_sender = MissionSender(
        queues_dir=queues_dir, client_id=car_id, log_level=LOG_ERROR)
    telemetry_sender = TelemetrySender(
        queues_dir=queues_dir, client_id=car_id, log_level=LOG_ERROR)

# Генерируем ключи для подписи событий
private_key_pem, public_key_pem = generate_rsa_keys()

# создание основных функциональных блоков
mission_planner = MissionPlanner(
    queues_dir, afcs_present=afcs_present, mission=mission)

sitl = SITL(
    queues_dir=queues_dir, position=home,
    car_id=car_id, post_telemetry=afcs_present, log_level=LOG_ERROR)

communication_gateway = CommunicationGateway(
    queues_dir=queues_dir, private_key=private_key, log_level=LOG_ERROR)
control_system = ControlSystem(queues_dir=queues_dir, private_key=private_key, log_level=LOG_INFO)

navigation_system = NavigationSystem(
    queues_dir=queues_dir, private_key=private_key, log_level=LOG_ERROR)

servos = Servos(queues_dir=queues_dir, log_level=LOG_ERROR)
cargo_bay = CargoBay(queues_dir=queues_dir, log_level=LOG_INFO)

safety_block = SafetyBlock(queues_dir=queues_dir, private_key=private_key,log_level=LOG_INFO)

security_monitor = SecurityMonitor(queues_dir=queues_dir, public_key=public_key, blackbox=blackbox)

# Создаем черный ящик
blackbox = BlackBox(public_key_pem=public_key)

# сборка всех запускаемых блоков в одном "кузове"
system_components = SystemComponentsContainer(
    components=[
        mission_sender,
        telemetry_sender,
        sitl,
        mission_planner,
        navigation_system,
        servos,
        cargo_bay,
        communication_gateway,
        control_system,
        safety_block,
        security_monitor, 
        blackbox
    ] if afcs_present else [
        sitl,
        mission_planner,
        navigation_system,
        servos,
        cargo_bay,
        communication_gateway,
        control_system,
        safety_block,
        security_monitor, 
        blackbox
    ])

#################################
# АКТИВАЦИЯ КИБЕРПРЕПЯТСТВИЙ
control_system.enable_surprises()
#################################

# запуск всех блоков
system_components.start()

# ограничение поездки по времени
# в случае превышения времени выполнения ячейки на более чем 10 секунд от заданного, 
# допустимо перезапустить вычислительное ядро и повторно выполнить весь блокнот, штрафные очки за это не начисляются
# при условии, что повторный запуск закончился успешно
sleep(750)

# останавливаем все компоненты
system_components.stop()

# удалим все созданные компоненты
system_components.clean()

[ИНФО][QUEUES] создан каталог очередей
[ИНФО][QUEUES] регистрируем очередь planner.mqtt
[ИНФО][QUEUES] регистрируем очередь sitl.mqtt
[ИНФО][QUEUES] регистрируем очередь planner
[ИНФО][MISSION PLANNER] создана система планирования заданий
[ИНФО][QUEUES] регистрируем очередь sitl
[ИНФО][QUEUES] регистрируем очередь communication
[ИНФО][QUEUES] регистрируем очередь control
[ИНФО][CONTROL] создана система управления
[ИНФО][QUEUES] регистрируем очередь navigation
[ИНФО][QUEUES] регистрируем очередь servos
[ИНФО][QUEUES] регистрируем очередь cargo
[ИНФО][CARGO] создан компонент грузового отсека, отсек заблокирован
[ИНФО][QUEUES] регистрируем очередь safety
[ИНФО][SAFETY] создан ограничитель
[ИНФО][QUEUES] регистрируем очередь security
[ИНФО][SECURITY] создан монитор безопасности
[ИНФО][SECURITY] изменение политик безопасности: [SecurityPolicy(source='communication', destination='control', operation='set_mission'), SecurityPolicy(source='communication', destination='safety', operation='set_m

Process CargoBay-10:
Process CommunicationGateway-6:
Process ControlSystem-7:
Process TelemetrySender-3:
Process SafetyBlock-11:
Process SecurityMonitor-12:
Process MissionPlanner-4:
Process NavigationSystem-8:
Process BlackBox-13:
Process SITL-5:
Process Servos-9:
Process MissionSender-2:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback 

KeyboardInterrupt: 

Убедитесь, что 
1. ваша машинка успешно прошла весь заданный маршрут
2. не превысила ограничения скорости
3. успешно доставила груз

Если всё так - поздравляем, вы справились с заданием! Обязательно зафиксируйте все изменения в репозитории!



Мы будем признательны за обратную связь - любые комментарии, которые вы можете дать по итогам выполнения этого задания. 

Например, 

- было ли задание понятным по шкале 1..10 (1 - ничего не понятно, 10 - вопросов вообще не было, всё понятно)?
- было ли задание интересным по шкале 1..10 (1 - скука смертная, 10 - лучшее, что вам пока встречалось на олимпиадах)? 
- что бы вы предложили изменить, чтобы сделает его более интересным?
- по шкале 1..10 насколько сложным оно было для вас?
- что было самым трудным в задании? 

Авторы наиболее развёрнутых и интересных комментариев получат особенный приз от Лаборатории Касперского!

Дополнительная информация о кибериммунной разработке
- https://os.kaspersky.ru/cyber-immune-development/ 
- https://github.com/sergey-sobolev/cyberimmune-systems/wiki/%D0%9A%D0%B8%D0%B1%D0%B5%D1%80%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D1%82%D0%B5%D1%82
- канал в телеграм: https://t.me/learning_cyberimmunity
